In [1]:
import pandas as pd

In [2]:
FILE_PATH = "data\landing\sms-call-internet-mi-2013-11-01.csv"
df = pd.read_csv(FILE_PATH)

<>:1: SyntaxWarning: invalid escape sequence '\l'
<>:1: SyntaxWarning: invalid escape sequence '\l'
C:\Users\yuvasree.r\AppData\Local\Temp\ipykernel_18488\2823696068.py:1: SyntaxWarning: invalid escape sequence '\l'
  FILE_PATH = "data\landing\sms-call-internet-mi-2013-11-01.csv"


In [3]:
print("Shape:", df.shape)
print("\nRaw Columns:", df.columns.tolist())
print("\nData Types:", df.dtypes)

Shape: (1891928, 8)

Raw Columns: ['datetime', 'CellID', 'countrycode', 'smsin', 'smsout', 'callin', 'callout', 'internet']

Data Types: datetime           str
CellID           int64
countrycode      int64
smsin          float64
smsout         float64
callin         float64
callout        float64
internet       float64
dtype: object


In [4]:
# Rename raw columns to canonical names
column_mapping = {
    "datetime": "timestamp",
    "CellID": "grid_id",
    "countrycode": "country_code",
    "smsin": "sms_in",
    "smsout": "sms_out",
    "callin": "call_in",
    "callout": "call_out",
    "internet": "internet_activity"
}

df = df.rename(columns=column_mapping)
print(df.columns.tolist())

['timestamp', 'grid_id', 'country_code', 'sms_in', 'sms_out', 'call_in', 'call_out', 'internet_activity']


In [25]:
#  Convert data types
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
activity_cols = [
    "sms_in",
    "sms_out",
    "call_in",
    "call_out",
    "internet_activity"
]
for col in activity_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")



In [8]:
# Time profiling
print("Missing timestamps:", df["timestamp"].isna().sum())

timestamps = sorted(df["timestamp"].dropna().unique())

print("Distinct timestamps:", len(timestamps))

if len(timestamps) > 0:
    print("Start:", timestamps[0])
    print("End:", timestamps[-1])

if len(timestamps) > 1:
    time_diff = pd.Series(timestamps).diff().dropna()

    print("Time intervals:")
    print(time_diff.value_counts())

    hourly_cadence = (time_diff == pd.Timedelta(hours=1)).all()

    print("Hourly cadence:", hourly_cadence)

Missing timestamps: 0
Distinct timestamps: 24
Start: 2013-11-01 00:00:00
End: 2013-11-01 23:00:00
Time intervals:
0 days 01:00:00    23
Name: count, dtype: int64
Hourly cadence: True


In [ ]:
df["date"] = df["timestamp"].dt.date
df["hour"] = df["timestamp"].dt.hour
df["day_of_week"] = df["timestamp"].dt.day_name()

In [10]:
print("\n========== NULL PROFILE ==========")

null_counts = df.isna().sum()

print(null_counts)

print("\nBlank activity values:")
print(df[activity_cols].isna().sum())


========== NULL PROFILE ==========
timestamp                  0
grid_id                    0
country_code               0
sms_in               1086153
sms_out              1422446
call_in              1407781
call_out             1037413
internet_activity    1087074
date                       0
hour                       0
day_of_week                0
dtype: int64

Blank activity values:
sms_in               1086153
sms_out              1422446
call_in              1407781
call_out             1037413
internet_activity    1087074
dtype: int64


In [11]:
print("\nNegative activity values:")

for col in activity_cols:
    count = (df[col] < 0).sum()
    print(f"{col}: {count}")


Negative activity values:
sms_in: 0
sms_out: 0
call_in: 0
call_out: 0
internet_activity: 0


In [12]:
duplicate_count = df.duplicated().sum()

print("\nExact duplicate rows:", duplicate_count)


Exact duplicate rows: 0


In [13]:
print("\n========== GRID PROFILE ==========")

print("Unique grids:", df["grid_id"].nunique())

print("Minimum grid_id:", df["grid_id"].min())
print("Maximum grid_id:", df["grid_id"].max())


========== GRID PROFILE ==========
Unique grids: 10000
Minimum grid_id: 1
Maximum grid_id: 10000


In [14]:
print("\n========== COUNTRY CODE PROFILE ==========")

print("Unique country codes:", df["country_code"].nunique())

print("\nCountry codes:")
print(df["country_code"].value_counts())


========== COUNTRY CODE PROFILE ==========
Unique country codes: 246

Country codes:
country_code
39      240000
0       231307
46      109848
33      106603
49       95933
         ...  
1938         4
247          3
240          2
596          2
1709         2
Name: count, Length: 246, dtype: int64


In [19]:
print("\n========== RAW GRAIN ==========")

grain_counts = (
    df.groupby(["timestamp", "grid_id"])["country_code"]
      .nunique() )
print(grain_counts)

multiple_country_rows = grain_counts[grain_counts > 1]

print("Grid + timestamp combinations having multiple country codes:",len(multiple_country_rows))

if len(multiple_country_rows) > 0:
    example = multiple_country_rows.index[0]

    example_rows = df[
        (df["timestamp"] == example[0]) &
        (df["grid_id"] == example[1])
    ]

    print("\nExample:")
    print(example_rows[
        ["timestamp", "grid_id", "country_code"]
    ])


========== RAW GRAIN ==========
timestamp            grid_id
2013-11-01 00:00:00  1          3
                     2          3
                     3          3
                     4          3
                     5          3
                               ..
2013-11-01 23:00:00  9996       6
                     9997       5
                     9998       5
                     9999       6
                     10000      6
Name: country_code, Length: 240000, dtype: int64
Grid + timestamp combinations having multiple country codes: 236949

Example:
   timestamp  grid_id  country_code
0 2013-11-01        1             0
1 2013-11-01        1            33
2 2013-11-01        1            39


In [20]:
df["total_sms"] = df["sms_in"] + df["sms_out"]

df["total_calls"] = df["call_in"] + df["call_out"]

df["total_activity"] = (
    df["total_sms"]
    + df["total_calls"]
    + df["internet_activity"]
)

In [21]:
# Busiest hourly window
hourly_activity = (
    df.groupby("timestamp")["total_activity"]
      .sum()
      .sort_values(ascending=False)
)

print("\n========== BUSIEST HOUR ==========")

print("Busiest timestamp:")
print(hourly_activity.index[0])

print("Total activity:")
print(hourly_activity.iloc[0])


========== BUSIEST HOUR ==========
Busiest timestamp:
2013-11-01 11:00:00
Total activity:
5009588.4424


In [22]:
# Busiest grid
grid_activity = (
    df.groupby("grid_id")["total_activity"]
      .sum()
      .sort_values(ascending=False)
)

print("\n========== BUSIEST GRID ==========")

print("Busiest grid:")
print(grid_activity.index[0])

print("Total activity:")
print(grid_activity.iloc[0])


========== BUSIEST GRID ==========
Busiest grid:
5161
Total activity:
262060.34


In [23]:
# Final profile summary
print("\n========================================")
print("           NP1 PROFILE SUMMARY")
print("========================================")

print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Unique grids:", df["grid_id"].nunique())
print("Distinct timestamps:", df["timestamp"].nunique())

if not df["timestamp"].isna().all():
    print("Time range:", df["timestamp"].min(), "to", df["timestamp"].max())

print("Country codes:", df["country_code"].nunique())
print("Exact duplicates:", duplicate_count)
print("Missing grid_id:", df["grid_id"].isna().sum())
print("Missing timestamp:", df["timestamp"].isna().sum())

print("\nNull counts:")
print(null_counts)


           NP1 PROFILE SUMMARY
Rows: 1891928
Columns: 14
Unique grids: 10000
Distinct timestamps: 24
Time range: 2013-11-01 00:00:00 to 2013-11-01 23:00:00
Country codes: 246
Exact duplicates: 0
Missing grid_id: 0
Missing timestamp: 0

Null counts:
timestamp                  0
grid_id                    0
country_code               0
sms_in               1086153
sms_out              1422446
call_in              1407781
call_out             1037413
internet_activity    1087074
date                       0
hour                       0
day_of_week                0
dtype: int64
